# Option Pricing Foundation

In [39]:
# import libraries
import numpy as np
from scipy.stats import norm
import yfinance as yf
import pandas as pd

## Helper Functions

In [29]:
def bs_call_price(S, K, T, r, sigma):
    """
    Black-Scholes Call Option Price (European)

    Args:
        S: current stock price
        K: strike price
        T: expiration date
        r: risk-free rate
        sigma: volatility

    Returns:
        C: call price
    """

    d1 = (np.log(S/K) + (r+0.5*sigma**2)*T)/(sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)

    C = S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)

    return C

In [30]:
def bs_put_price(S, K, T, r, sigma):
    """
    Black-Scholes Put Option Price (European)

    Args:
        S: current stock price
        K: strike price
        T: expiration date
        r: risk-free rate
        sigma: volatility

    Returns:
        P: put price
    """
    
    d1 = (np.log(S/K) + (r+0.5*sigma**2)*T)/(sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)

    P = K*np.exp(-r*T)*norm.cdf(-d2) - S*norm.cdf(-d1)

    return P

In [31]:
def get_delta(S, K, T, r, sigma, call = True):
    """
    Get delta of option pricing

    Args:
        S: current stock price
        K: strike price
        T: expiration date
        r: risk-free rate
        sigma: volatility

    Returns:
        delta: sensitivity to stock price
    """

    d1 = (np.log(S/K) + (r+0.5*sigma**2)*T)/(sigma*np.sqrt(T))

    if call:
        delta = norm.cdf(d1)
    else:
        delta = norm.cdf(d1) - 1

    return delta

In [32]:
def get_gamma(S, K, T, r, sigma):
    """
    Get gamma of option pricing

    Args:
        S: current stock price
        K: strike price
        T: expiration date
        r: risk-free rate
        sigma: volatility

    Returns:
        gamma: sensitivity of delta to stock price
    """

    d1 = (np.log(S/K) + (r+0.5*sigma**2)*T)/(sigma*np.sqrt(T))

    gamma = (norm.pdf(d1))/(S*sigma*np.sqrt(T))

    return gamma

In [33]:
def get_vega(S, K, T, r, sigma):
    """
    Get vega of option pricing

    Args:
        S: current stock price
        K: strike price
        T: expiration date
        r: risk-free rate
        sigma: volatility

    Returns:
        vega: sensitivity to volatility
    """

    d1 = (np.log(S/K) + (r+0.5*sigma**2)*T)/(sigma*np.sqrt(T))

    vega = S*norm.pdf(d1)*np.sqrt(T)

    return round(vega/100, 2)

In [34]:
def get_theta(S, K, T, r, sigma, call=True):
    """
    Get theta of option pricing

    Args:
        S: current stock price
        K: strike price
        T: expiration date
        r: risk-free rate
        sigma: volatility

    Returns:
        theta: sensitivity to expiration date
    """

    d1 = (np.log(S/K) + (r+0.5*sigma**2)*T)/(sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)

    if call:
        theta = -(S*norm.pdf(d1)*sigma)/(2*np.sqrt(T)) - r*K*np.exp(-r*T)*norm.cdf(d2)
    else:
        theta = -(S*norm.pdf(d1)*sigma)/(2*np.sqrt(T)) + r*K*np.exp(-r*T)*norm.cdf(-d2)

    return theta

In [35]:
def get_rho(S, K, T, r, sigma, call=True):
    """
    Get rho of option pricing

    Args:
        S: current stock price
        K: strike price
        T: expiration date
        r: risk-free rate
        sigma: volatility

    Returns:
        rho: sensitivity to risk-free rate
    """

    d1 = (np.log(S/K) + (r+0.5*sigma**2)*T)/(sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)

    if call:
        rho = K*T*np.exp(-r*T)*norm.cdf(d2)
    else:
        rho = -K*T*np.exp(-r*T)*norm.cdf(-d2)

    return round(rho/100, 2)

In [36]:
# Example parameters
S = 100      # stock price
K = 105      # strike price
T = 1        # 1 year to maturity
r = 0.05     # 5% risk-free interest rate
sigma = 0.2  # 20% volatility

call = bs_call_price(S, K, T, r, sigma)
put = bs_put_price(S, K, T, r, sigma)

print(f"Call Option Price: {call:.4f}")
print(f"Put Option Price: {put:.4f}")

delta_call = get_delta(S, K, T, r, sigma, call = True)
print(f"delta of call price: {delta_call}")

delta_put = get_delta(S, K, T, r, sigma, call = False)
print(f"delta of put price: {delta_put}")

gamma = get_gamma(S, K, T, r, sigma)
print(f"gamma: {gamma}")

vega = get_vega(S, K, T, r, sigma)
print(f"vega: {vega}")

theta_call = get_theta(S, K, T, r, sigma, call=True)
print(f"theta of call price: {theta_call}")

theta_put = get_theta(S, K, T, r, sigma, call=False)
print(f"theta of put price: {theta_put}")

rho_call = get_rho(S, K, T, r, sigma, call=True)
print(f"rho of call price: {rho_call}")

rho_put = get_rho(S, K, T, r, sigma, call=False)
print(f"rho of put price: {rho_put}")

Call Option Price: 8.0214
Put Option Price: 7.9004
delta of call price: 0.5422283335848053
delta of put price: -0.45777166641519473
gamma: 0.019835261904213263
vega: 0.4
theta of call price: -6.277126437009521
theta of put price: -1.283171958380772
rho of call price: 0.46
rho of put price: -0.54


In [37]:
eps = 1e-4
rhs = (bs_call_price(S+eps, K, T, r, sigma) - bs_call_price(S, K, T, r, sigma)
)/eps
lhs = delta_call

print(f"LHS: {lhs} ~  RHS: {rhs}")

LHS: 0.5422283335848053 ~  RHS: 0.5422293254042643


## Validating BS Model

In [38]:
ticker = yf.Ticker("TSLA")

expirations = ticker.options
print(f"Available expirations: {expirations}")

expiry = expirations[0] # picking nearest expiry

opt_chain = ticker.option_chain(expiry)
calls = opt_chain.calls
puts = opt_chain.puts

Available expirations: ('2025-09-05', '2025-09-12', '2025-09-19', '2025-09-26', '2025-10-03', '2025-10-10', '2025-10-17', '2025-11-21', '2025-12-19', '2026-01-16', '2026-02-20', '2026-03-20', '2026-04-17', '2026-05-15', '2026-06-18', '2026-07-17', '2026-08-21', '2026-09-18', '2026-12-18', '2027-01-15', '2027-06-17', '2027-12-17')


In [46]:
S = ticker.history(period="1d")["Close"].iloc[-1]  # current stock price
T = (pd.to_datetime(expiry) - pd.Timestamp.today()).days / 365
# assumptions
r = 0.05
sigma = 0.1

In [47]:
for i, row in calls.head(20).iterrows():
    K = row["strike"]
    market_price = row["lastPrice"]
    bs_price = bs_call_price(S, K, T, r, sigma)
    print(f"Strike={K}\tMarket={market_price}\tBS={bs_price:.2f}")

Strike=50.0	Market=264.0	BS=283.89
Strike=100.0	Market=233.47	BS=233.91
Strike=120.0	Market=212.83	BS=213.92
Strike=130.0	Market=202.77	BS=203.92
Strike=135.0	Market=215.25	BS=198.93
Strike=140.0	Market=197.75	BS=193.93
Strike=145.0	Market=187.3	BS=188.93
Strike=150.0	Market=186.26	BS=183.93
Strike=155.0	Market=167.7	BS=178.93
Strike=160.0	Market=175.42	BS=173.94
Strike=165.0	Market=145.6	BS=168.94
Strike=170.0	Market=162.39	BS=163.94
Strike=175.0	Market=139.1	BS=158.94
Strike=180.0	Market=169.46	BS=153.94
Strike=185.0	Market=165.3	BS=148.95
Strike=190.0	Market=155.23	BS=143.95
Strike=195.0	Market=156.09	BS=138.95
Strike=200.0	Market=132.75	BS=133.95
Strike=205.0	Market=130.53	BS=128.95
Strike=210.0	Market=122.59	BS=123.96


In [48]:
for i, row in puts.head(20).iterrows():
    K = row["strike"]
    market_price = row["lastPrice"]
    bs_price = bs_put_price(S, K, T, r, sigma)
    print(f"Strike={K}\tMarket={market_price}\tBS={bs_price:.2f}")

Strike=50.0	Market=0.01	BS=0.00
Strike=60.0	Market=0.01	BS=0.00
Strike=80.0	Market=0.01	BS=0.00
Strike=90.0	Market=0.01	BS=0.00
Strike=100.0	Market=0.01	BS=0.00
Strike=110.0	Market=0.01	BS=0.00
Strike=120.0	Market=0.01	BS=0.00
Strike=130.0	Market=0.01	BS=0.00
Strike=135.0	Market=0.01	BS=0.00
Strike=140.0	Market=0.02	BS=0.00
Strike=145.0	Market=0.01	BS=0.00
Strike=150.0	Market=0.01	BS=0.00
Strike=155.0	Market=0.01	BS=0.00
Strike=160.0	Market=0.02	BS=0.00
Strike=165.0	Market=0.03	BS=0.00
Strike=170.0	Market=0.04	BS=0.00
Strike=175.0	Market=0.02	BS=0.00
Strike=180.0	Market=0.02	BS=0.00
Strike=185.0	Market=0.04	BS=0.00
Strike=190.0	Market=0.03	BS=0.00
